In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# --- 1. Define Classification Logic (Existing) ---
def classify_component(smiles):
    if pd.isna(smiles) or smiles == "":
        return "Invalid", "Empty"
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid", "RDKit parse error"

    # Matches: Alpha-Beta Unsaturated Ester
    acrylate_pattern = Chem.MolFromSmarts('[CX3]=[CX3][CX3](=[OX1])[OX2]')
    
    if not mol.HasSubstructMatch(acrylate_pattern):
        return "Not Acrylate", "No acrylic moiety"

    matches = mol.GetSubstructMatches(acrylate_pattern)
    found_types = set()
    
    for match in matches:
        beta_idx, alpha_idx, carbonyl_idx = match[0], match[1], match[2]
        beta_atom = mol.GetAtomWithIdx(beta_idx)
        alpha_atom = mol.GetAtomWithIdx(alpha_idx)
        
        if beta_atom.GetTotalNumHs() == 2: # Terminal Alkene
            if alpha_atom.GetTotalNumHs() == 1:
                found_types.add("Acrylate")
            else:
                is_methyl = False
                for neighbor in alpha_atom.GetNeighbors():
                    if neighbor.GetIdx() not in [beta_idx, carbonyl_idx]:
                        if neighbor.GetAtomicNum() == 6 and neighbor.GetTotalNumHs() == 3:
                            is_methyl = True
                if is_methyl:
                    found_types.add("Methacrylate")
                else:
                    found_types.add("Alpha-Substituted Acrylate")
        else:
            found_types.add("Beta-Substituted")

    if not found_types: return "Uncertain", "Pattern matched but structure unclear"
    if "Acrylate" in found_types and "Methacrylate" in found_types: return "Mixed Acrylate/Methacrylate", "Contains both moieties"
    if len(found_types) == 1: return list(found_types)[0], f"Found {list(found_types)[0]} moiety"
    return "Mixed Types", f"Found: {', '.join(found_types)}"

# --- 2. Define Transformation Logic (New) ---
# Pre-compile the reaction to improve performance
# Logic: Find terminal alkene with alpha-carbon substituent -> Replace with terminal alkene with alpha-hydrogen
# Note: The atom mapping (:1, :2...) preserves the connectivity of the ester tail.
RXN_SMARTS = '[CH2:1]=[C:2]([#6])[C:3](=[O:4])[O:5]>>[CH2:1]=[CH:2][C:3](=[O:4])[O:5]'
ACRYLATE_RXN = AllChem.ReactionFromSmarts(RXN_SMARTS)

def standardize_to_acrylate(smiles):
    """
    Converts Methacrylates and Alpha-Substituted Acrylates into standard Acrylates
    by removing the substituent at the alpha position.
    """
    if pd.isna(smiles) or smiles == "": return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None

    # We use a loop to handle molecules with MULTIPLE reactive sites (e.g., crosslinkers)
    # RDKit's RunReactants applies the transformation once per call.
    current_mol = mol
    transformation_happened = False
    
    while True:
        # Run the reaction on the current molecule state
        products = ACRYLATE_RXN.RunReactants((current_mol,))
        
        # If no products, it means no more patterns matched
        if not products:
            break
            
        # Take the first product of the first match (modify the molecule)
        current_mol = products[0][0]
        transformation_happened = True
        
        # Sanitize to update valence/aromaticity after modification
        try:
            Chem.SanitizeMol(current_mol)
        except:
            return "Error during sanitization"

    if transformation_happened:
        return Chem.MolToSmiles(current_mol)
    else:
        # If no transformation occurred, return the original SMILES
        return smiles

# --- 3. Execution Pipeline ---

# Load Data
df = pd.read_csv('../polygraphpy/data/original_dataset.csv')

# Separate Mixtures
df['smiles_component'] = df['smiles'].str.split('.')
df_separated = df.explode('smiles_component')

# A. Classify
print("Classifying components...")
results = df_separated['smiles_component'].apply(classify_component)
df_separated['Category'] = [r[0] for r in results]
df_separated['Reason'] = [r[1] for r in results]

# B. Transform (New Step)
print("Standardizing acrylates...")
df_separated['smiles_normalized'] = df_separated['smiles_component'].apply(standardize_to_acrylate)

# --- 4. Save Result ---
# We keep both the original component SMILES and the normalized version
output_cols = ['id', 'cmpdname', 'smiles_component', 'smiles_normalized', 'mw','mf','polararea','complexity', 'Category', 'Reason']

# Check if 'id' and 'cmpdname' exist, otherwise adjust columns
available_cols = [c for c in output_cols if c in df_separated.columns]
df_final = df_separated[available_cols].drop_duplicates(subset='smiles_component')

Classifying components...


[12:02:28] WARNING: not removing hydrogen atom without neighbors


Standardizing acrylates...


[12:02:28] WARNING: not removing hydrogen atom without neighbors


In [3]:
results = df_final['smiles_normalized'].apply(classify_component)
df_final['Category_renormalized'] = [r[0] for r in results]
df_final['Reason_renormalized'] = [r[1] for r in results]
df_final = df_final.drop_duplicates(subset='smiles_normalized').reset_index(drop=True)

df_final.to_csv('acrylates_classified.csv', index=False)

[12:02:44] WARNING: not removing hydrogen atom without neighbors


In [5]:
input_df = df_final[df_final['Category_renormalized'] == 'Acrylate'].reset_index(drop=True)

In [7]:
input_df[['id', 'cmpdname', 'smiles_normalized', 'mw', 'mf', 'polararea', 'complexity']].rename(columns={'smiles_normalized':'smiles'}).to_csv('../polygraphpy/data/full_dataset.csv')